## Demo of client side Union of hotset dataset as view over Kafka through ISK and coldset dataset on MiniIO

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Jupyter").getOrCreate()

spark

25/09/25 12:36:32 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.




### ISK view over Kafka has multiple topics including two topics with transactions data:
    
 - **transactions_old** with static set of data of 500000 events sorted by TransactionTime - that can be loaded into the cold set on MiniIO for demo purposes
 - **transactions** with dynamic set of data - starting with 50 events and gets new event every second.



In [2]:
%%sql

USE isk.isk

++
||
++
++

In [3]:
%%sql

show tables;

namespace,tableName,isTemporary
isk,transactions_old,False
isk,accounts,False
isk,customers,False
isk,branches,False
isk,transactions,False


In [4]:
%%sql

DESCRIBE transactions_old;


col_name,data_type,comment
TransactionID,string,None
AccountID,string,None
TransactionType,string,None
TransactionAmount,double,None
BranchID,bigint,None
FraudRiskScore,double,None
TransactionTime,timestamp_ntz,None


In [5]:
%%sql

SELECT COUNT(*) FROM transactions_old;

count(1)
500000


In [ ]:
#
# We've started in the middle, our hotset is too large and we must move it to cold storage
#

In [6]:
%%sql
--  Load transactions_old events into the coldset using CTAS operation, partitioned by TransactionTime hour.
CREATE TABLE minio.data.transactions
          USING iceberg
          PARTITIONED BY (HOUR(TransactionTime))
          TBLPROPERTIES('format-version'='2')
 AS SELECT * FROM isk.isk.transactions_old;

25/09/25 12:38:03 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
25/09/25 12:38:04 WARN S3ABlockOutputStream: Application invoked the Syncable API against stream writing to data/transactions/data/TransactionTime_hour=2025-08-22-14/00001-253-33f7d9f4-2e81-41f0-8239-93a06ca1cdf9-0-00006.parquet. This is unsupported
                                                                                

++
||
++
++

In [7]:
# Delete transactions_old from Kafka

In [8]:
%%sql
-- Verify number of rows written to the cold set
select count (*) from  minio.data.transactions;

count(1)
500000


In [9]:
# Show data now in minio

In [10]:
#
# Now let's move on to the hotset, this shows incoming data from Kafka
#

In [13]:
%%sql
-- Verify number of rows on the hot set in transactions topic
select count (*) from  isk.isk.transactions where transactionTime < Now();

count(1)
368


In [ ]:
#
# We can union our hotset and coldset data to create the super powerful mixed applications
#

In [14]:
%%sql
-- Count of transaction events across cold and hot set 

select count (*) from (SELECT * FROM minio.data.transactions m
UNION
select * from isk.isk.transactions i WHERE i.transactionTime> (SELECT max(transactionTime) FROM minio.data.transactions));

count(1)
500390


In [15]:
# Run the count again to show new data seamlessly available

In [16]:
#
# A more complex application
#

In [17]:
%%sql
-- Total deposits - withdrawals per branch - without time bounds across cold and hotset   
    
select round(sum( 
case 
    when t.transactiontype='Withdrawal' THEN t.transactionamount*(-1)
    else t.transactionamount
END
),2) as total_per_branch,
b.branchname  
from 
(SELECT * FROM minio.data.transactions m
UNION
select * from isk.isk.transactions i) t    
    
    join branches b on t.branchid=b.branchid 
--    where t.transactiontime between '2025-04-21 00:00:00' AND '2025-04-21 23:59:59'
    group by b.branchname order by b.branchname asc;

total_per_branch,branchname
671285.47,Bashirian Group
87825.52,"Boyer, Will and Nienow"
-190942.52,Champlin-Weissnat
186458.6,Cormier-Jacobson
157778.65,"Cummings, Weber and Trantow"
808555.04,Emmerich-Kovacek
-448773.44,Erdman-Ortiz
-850680.84,Fadel LLC
-967956.25,Franecki and Sons
467556.67,Grady-Rolfson


In [ ]:
# Optional: show new data coming in

In [ ]:
%%sql
-- Latest 100 events on the hotset

select * from isk.isk.transactions ORDER BY transactionTime DESC LIMIT 100;